In [15]:
%pip install -q pandas numpy pillow matplotlib scikit-learn torch torchvision tqdm opencv-python

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "deep_learning_recognition" else Path.cwd()
LABELS_PATH = PROJECT_ROOT / "label_tool" / "labels.csv"
NOTE_DIR = PROJECT_ROOT / "data" / "note_pitches_selected"
SYMBOL_DIR = PROJECT_ROOT / "data" / "symbol_types_selected"

df = pd.read_csv(LABELS_PATH)

df.head()

,filename,sym_type,note_pitch
0,1_1.png,K:t,-
1,1_10.png,8,B4
2,1_11.png,8,B4
3,1_12.png,8,C5
4,1_13.png,8,B4


In [2]:
print(f"Rows in labels.csv: {len(df)}")
print(f"Unique sym_type: {df['sym_type'].nunique()}")
print(f"Unique note_pitch: {df['note_pitch'].nunique()}")
print(f"NOTE_DIR exists: {NOTE_DIR.exists()}")
print(f"SYMBOL_DIR exists: {SYMBOL_DIR.exists()}")

Rows in labels.csv: 658
Unique sym_type: 26
Unique note_pitch: 25
NOTE_DIR exists: True
SYMBOL_DIR exists: True


## Etap 1: Przygotowanie etykiet i podział train/val
Budujemy mapowanie klas dla `sym_type` i `note_pitch`, a potem dzielimy dane na zbiory treningowy i walidacyjny.

In [4]:
from sklearn.model_selection import train_test_split

work_df = df.copy()
work_df["sym_type"] = work_df["sym_type"].fillna("UNK").astype(str)
work_df["note_pitch"] = work_df["note_pitch"].fillna("-").astype(str)

# Zostawiamy tylko rekordy, dla których istnieją oba obrazy.
def has_pair(filename: str) -> bool:
    return (NOTE_DIR / filename).exists() and (SYMBOL_DIR / filename).exists()

work_df = work_df[work_df["filename"].map(has_pair)].reset_index(drop=True)

sym_classes = sorted(work_df["sym_type"].unique().tolist())
pitch_classes = sorted(work_df["note_pitch"].unique().tolist())

sym_to_idx = {c: i for i, c in enumerate(sym_classes)}
pitch_to_idx = {c: i for i, c in enumerate(pitch_classes)}
idx_to_sym = {i: c for c, i in sym_to_idx.items()}
idx_to_pitch = {i: c for c, i in pitch_to_idx.items()}

train_df, val_df = train_test_split(
    work_df,
    test_size=0.2,
    random_state=42,
    stratify=work_df["sym_type"] if work_df["sym_type"].nunique() > 1 else None,
)

print(f"Filtered rows: {len(work_df)}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"sym_type classes: {len(sym_classes)}")
print(f"note_pitch classes: {len(pitch_classes)}")

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

## Etap 2: Dataset i DataLoadery
Tworzymy dataset z dwoma kanałami wejściowymi: kanał 1 to obraz z `note_pitches_selected`, kanał 2 to obraz z `symbol_types_selected`.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMG_SIZE = 64
BATCH_SIZE = 64

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.ToTensor(),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])


class OMRPairDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _load_gray(self, path: Path):
        img = Image.open(path).convert("L")
        if self.transform is not None:
            img = self.transform(img)
        else:
            img = T.ToTensor()(img)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row["filename"]

        note_img = self._load_gray(NOTE_DIR / filename)
        sym_img = self._load_gray(SYMBOL_DIR / filename)

        x = torch.cat([note_img, sym_img], dim=0)  # [2, H, W]
        y_sym = sym_to_idx[row["sym_type"]]
        y_pitch = pitch_to_idx[row["note_pitch"]]

        return x, torch.tensor(y_sym), torch.tensor(y_pitch), filename


train_ds = OMRPairDataset(train_df, transform=train_tf)
val_ds = OMRPairDataset(val_df, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, ysb, ypb, names = next(iter(train_loader))
print("Batch x shape:", xb.shape)
print("Batch sym shape:", ysb.shape, "Batch pitch shape:", ypb.shape)
print("Example filename:", names[0])

## Etap 3: Model multi-head (sym_type + note_pitch)
Backbone CNN zwraca wspólne cechy, a dwie głowice klasyfikują odpowiednio typ symbolu i wysokość nuty.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.net(x)


class MultiHeadOMRNet(nn.Module):
    def __init__(self, in_channels=2, num_sym=1, num_pitch=1):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.2)
        self.sym_head = nn.Linear(256, num_sym)
        self.pitch_head = nn.Linear(256, num_pitch)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        sym_logits = self.sym_head(x)
        pitch_logits = self.pitch_head(x)
        return sym_logits, pitch_logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiHeadOMRNet(
    in_channels=2,
    num_sym=len(sym_classes),
    num_pitch=len(pitch_classes),
).to(device)

model

## Etap 4: Trening
Uczymy model joint loss: `L = L_sym + 0.7 * L_pitch`.

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 8
LR = 1e-3
W_PITCH = 0.7

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def run_epoch(loader, train_mode=True):
    model.train(train_mode)
    total_loss = 0.0
    total_sym_correct = 0
    total_pitch_correct = 0
    total_count = 0

    for x, y_sym, y_pitch, _ in tqdm(loader, leave=False):
        x = x.to(device)
        y_sym = y_sym.to(device)
        y_pitch = y_pitch.to(device)

        with torch.set_grad_enabled(train_mode):
            sym_logits, pitch_logits = model(x)
            loss_sym = criterion(sym_logits, y_sym)
            loss_pitch = criterion(pitch_logits, y_pitch)
            loss = loss_sym + W_PITCH * loss_pitch

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_sym_correct += (sym_logits.argmax(1) == y_sym).sum().item()
        total_pitch_correct += (pitch_logits.argmax(1) == y_pitch).sum().item()
        total_count += x.size(0)

    return {
        "loss": total_loss / total_count,
        "sym_acc": total_sym_correct / total_count,
        "pitch_acc": total_pitch_correct / total_count,
    }


history = {"train_loss": [], "val_loss": [], "train_sym_acc": [], "val_sym_acc": [], "train_pitch_acc": [], "val_pitch_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train_mode=True)
    val_metrics = run_epoch(val_loader, train_mode=False)

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["train_sym_acc"].append(train_metrics["sym_acc"])
    history["val_sym_acc"].append(val_metrics["sym_acc"])
    history["train_pitch_acc"].append(train_metrics["pitch_acc"])
    history["val_pitch_acc"].append(val_metrics["pitch_acc"])

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, sym={train_metrics['sym_acc']:.3f}, pitch={train_metrics['pitch_acc']:.3f} | "
        f"val loss={val_metrics['loss']:.4f}, sym={val_metrics['sym_acc']:.3f}, pitch={val_metrics['pitch_acc']:.3f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["train_sym_acc"], label="train")
axes[1].plot(history["val_sym_acc"], label="val")
axes[1].set_title("sym_type accuracy")
axes[1].legend()

axes[2].plot(history["train_pitch_acc"], label="train")
axes[2].plot(history["val_pitch_acc"], label="val")
axes[2].set_title("note_pitch accuracy")
axes[2].legend()

plt.tight_layout()
plt.show()

## Etap 5: Sliding-window inference na całej pięciolinii
Skanujemy fragment nutowy oknami o różnych skalach, klasyfikujemy każdy patch i stosujemy NMS, aby usunąć duplikaty.

In [ ]:
import cv2
from typing import List, Dict


def preprocess_staff_image(path: Path, target_height: int = 128):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")

    h, w = img.shape
    scale = target_height / h
    resized = cv2.resize(img, (int(w * scale), target_height), interpolation=cv2.INTER_AREA)
    return resized


def sliding_windows(img: np.ndarray, window_sizes=(24, 32, 40, 48), stride=6):
    h, w = img.shape
    windows = []
    for ws in window_sizes:
        for y in range(0, max(1, h - ws + 1), stride):
            for x in range(0, max(1, w - ws + 1), stride):
                crop = img[y:y + ws, x:x + ws]
                if crop.shape[0] != ws or crop.shape[1] != ws:
                    continue
                windows.append((x, y, ws, ws, crop))
    return windows


def iou_xywh(a, b):
    ax1, ay1, aw, ah = a
    bx1, by1, bw, bh = b
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter = inter_w * inter_h

    area_a = aw * ah
    area_b = bw * bh
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def nms(dets: List[Dict], iou_thr=0.3):
    dets = sorted(dets, key=lambda d: d["score"], reverse=True)
    kept = []
    for d in dets:
        keep = True
        for k in kept:
            if iou_xywh(d["bbox"], k["bbox"]) > iou_thr:
                keep = False
                break
        if keep:
            kept.append(d)
    return kept


@torch.no_grad()
def detect_symbols_sliding_window(
    model,
    image_path: Path,
    conf_thr=0.85,
    stride=6,
    window_sizes=(24, 32, 40, 48),
):
    model.eval()

    gray = preprocess_staff_image(image_path)
    windows = sliding_windows(gray, window_sizes=window_sizes, stride=stride)

    detections = []
    for x, y, w, h, crop in windows:
        crop_resized = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        patch = torch.tensor(crop_resized / 255.0, dtype=torch.float32).unsqueeze(0)
        x_in = torch.cat([patch, patch], dim=0).unsqueeze(0).to(device)  # [1,2,H,W]

        sym_logits, pitch_logits = model(x_in)
        sym_prob = F.softmax(sym_logits, dim=1)[0]
        pitch_prob = F.softmax(pitch_logits, dim=1)[0]

        sym_conf, sym_idx = torch.max(sym_prob, dim=0)
        pitch_conf, pitch_idx = torch.max(pitch_prob, dim=0)

        score = float(sym_conf * pitch_conf)
        if score < conf_thr:
            continue

        detections.append(
            {
                "bbox": (x, y, w, h),
                "score": score,
                "sym_type": idx_to_sym[int(sym_idx)],
                "note_pitch": idx_to_pitch[int(pitch_idx)],
            }
        )

    detections = nms(detections, iou_thr=0.3)
    detections = sorted(detections, key=lambda d: d["bbox"][0])

    return gray, detections

In [ ]:
# Podaj ścieżkę do obrazu całego fragmentu nutowego.
# Przykład:
# STAFF_IMAGE_PATH = PROJECT_ROOT / "data" / "my_staff_fragment.png"

STAFF_IMAGE_PATH = PROJECT_ROOT / "data" / "my_staff_fragment.png"

if STAFF_IMAGE_PATH.exists():
    gray, dets = detect_symbols_sliding_window(
        model,
        STAFF_IMAGE_PATH,
        conf_thr=0.90,
        stride=6,
        window_sizes=(24, 32, 40, 48),
    )

    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    for d in dets:
        x, y, w, h = d["bbox"]
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 180, 0), 1)
        label = f"{d['sym_type']}|{d['note_pitch']}:{d['score']:.2f}"
        cv2.putText(vis, label, (x, max(10, y - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 180, 0), 1)

    plt.figure(figsize=(18, 4))
    plt.imshow(vis[:, :, ::-1])
    plt.axis("off")
    plt.title("Sliding-window detections")
    plt.show()

    print("Sequence left-to-right:")
    print([f"{d['sym_type']}:{d['note_pitch']}" for d in dets])
else:
    print("Ustaw STAFF_IMAGE_PATH na istniejący obraz pięciolinii.")